# Simulate the exposure->conversion paths

In [43]:
import numpy as np
import pandas as pd
from typing import Dict
from collections import defaultdict

RNG = np.random.default_rng(42)

CHANNELS = ["Display", "Search", "Social", "Email"]
START = "Start"
CONVERSION = "Conversion"
NULL = "Null"

ALL_STATES = [START] + CHANNELS + [CONVERSION, NULL]


def simulate_paths(
    n_paths: int = 500_000,
    max_steps: int = 5,
    rng: np.random.Generator | None = None,
    terminal_probs: Dict = {
        "Display": {"Conversion": 0.009, "Null": 0.1},
        "Search":  {"Conversion": 0.0276,  "Null": 0.75},
        "Social":  {"Conversion": 0.008, "Null": 0.1},
        "Email":   {"Conversion": 0.0135,  "Null": 0.20},
    },
    trans_probs: Dict = {
        "Display": {
            "Display": 0.08,
            "Search": 0.45,
            "Social": 0.25,
            "Email": 0.22,
        },
    
        "Social": {
            "Display": 0.18,
            "Search": 0.50,
            "Social": 0.07,
            "Email": 0.25,
        },
    
        "Search": {
            "Display": 0.02,
            "Search": 0.18,
            "Social": 0.02,
            "Email": 0.78,
        },
    
        "Email": {
            "Display": 0.02,
            "Search": 0.60,
            "Social": 0.03,
            "Email": 0.35,
        },
    }
) -> pd.DataFrame:
    """
    Simulate user journeys for Markov attribution.

    Returns a dataframe with columns:
      - path_id
      - path (list of states excluding Start, including terminal state)
      - converted (0/1)
    """
    def normalize_probs(prob_dict: dict[str, float]) -> dict[str, float]:
        total = sum(prob_dict.values())
        if total <= 0:
            raise ValueError("Probability row sum must be positive.")
        return {k: v / total for k, v in prob_dict.items()}    
    
    if rng is None:
        rng = np.random.default_rng(42)

    # Initial channel distribution from Start
    start_probs = {
        "Display": 0.45,
        "Search": 0.06,
        "Social": 0.33,
        "Email": 0.16,
    }
    # Channel-to-channel transitions
    # trans_probs = {
    #     "Display": normalize_probs({"Display": 0.10, "Search": 0.30, "Social": 0.25, "Email": 0.15}),
    #     "Search":  normalize_probs({"Display": 0.10, "Search": 0.15, "Social": 0.10, "Email": 0.20}),
    #     "Social":  normalize_probs({"Display": 0.25, "Search": 0.20, "Social": 0.15, "Email": 0.15}),
    #     "Email":   normalize_probs({"Display": 0.05, "Search": 0.35, "Social": 0.05, "Email": 0.10}),
    # }
    # trans_probs = {
    #     "Display": {
    #         "Display": 0.05,
    #         "Search":  0.75,
    #         "Social":  0.05,
    #         "Email":   0.15,
    #     },
    #     "Search": {
    #         "Display": 0.03,
    #         "Search":  0.15,
    #         "Social":  0.02,
    #         "Email":   0.80,
    #     },
    #     "Social": {
    #         "Display": 0.05,
    #         "Search":  0.75,
    #         "Social":  0.05,
    #         "Email":   0.15,
    #     },
    #     "Email": {
    #         "Display": 0.02,
    #         "Search":  0.60,
    #         "Social":  0.03,
    #         "Email":   0.35,
    #     },
    # }
    # Terminal probabilities after visiting each channel
    # Remaining probability continues to another channel.
    # terminal_probs = {
    #     "Display": {"Conversion": 0.007, "Null": 0.02},
    #     "Search":  {"Conversion": 0.024,  "Null": 0.1},
    #     "Social":  {"Conversion": 0.008, "Null": 0.01},
    #     "Email":   {"Conversion": 0.020,  "Null": 0.1},
    # }

    # terminal_probs = {
    #     "Display": {"Conversion": 0.009, "Null": 0.1},
    #     "Search":  {"Conversion": 0.0276,  "Null": 0.7},
    #     "Social":  {"Conversion": 0.008, "Null": 0.1},
    #     "Email":   {"Conversion": 0.0135,  "Null": 0.20},
    # }

    rows = []

    for path_id in range(n_paths):
        path = []
        current = START

        # first touch
        first_channel = rng.choice(
            list(start_probs.keys()),
            p=list(start_probs.values())
        )
        current = first_channel
        path.append(current)

        for _ in range(max_steps):
            conv_p = terminal_probs[current]["Conversion"]
            null_p = terminal_probs[current]["Null"]
            cont_p = 1.0 - conv_p - null_p

            outcome = rng.choice(
                ["Conversion", "Null", "Continue"],
                p=[conv_p, null_p, cont_p]
            )

            if outcome == "Conversion":
                path.append(CONVERSION)
                break
            elif outcome == "Null":
                path.append(NULL)
                break
            else:
                next_channel_probs = trans_probs[current]
                next_channel = rng.choice(
                    list(next_channel_probs.keys()),
                    p=list(next_channel_probs.values())
                )
                current = next_channel
                path.append(current)
        else:
            # if max_steps reached without termination, force null
            path.append(NULL)

        converted = int(path[-1] == CONVERSION)
        rows.append(
            {
                "path_id": path_id,
                "path": path,
                "converted": converted,
            }
        )

    return pd.DataFrame(rows)

# Convert those paths into a Markov chain

In [44]:
def build_transition_counts(paths_df: pd.DataFrame) -> pd.DataFrame:
    """
    Count transitions including Start -> first_channel and channel -> terminal.
    """
    counts = defaultdict(int)

    for path in paths_df["path"]:
        full_path = [START] + path
        for a, b in zip(full_path[:-1], full_path[1:]):
            counts[(a, b)] += 1

    rows = [{"from_state": a, "to_state": b, "count": c} for (a, b), c in counts.items()]
    return pd.DataFrame(rows)


def build_transition_matrix(count_df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize counts row-wise to get transition probabilities.
    """
    df = count_df.copy()
    df["row_sum"] = df.groupby("from_state")["count"].transform("sum")
    df["prob"] = df["count"] / df["row_sum"]

    mat = df.pivot(index="from_state", columns="to_state", values="prob").fillna(0.0)

    # ensure all states exist
    for s in ALL_STATES:
        if s not in mat.index:
            mat.loc[s] = 0.0
        if s not in mat.columns:
            mat[s] = 0.0

    mat = mat.loc[ALL_STATES, ALL_STATES]

    # absorbing states
    mat.loc[CONVERSION, :] = 0.0
    mat.loc[CONVERSION, CONVERSION] = 1.0
    mat.loc[NULL, :] = 0.0
    mat.loc[NULL, NULL] = 1.0

    return mat

# Compute overall conversion probability from Start

In [45]:
# def conversion_probability_from_start(P: pd.DataFrame) -> float:
#     """
#     Compute absorption probability into Conversion starting from Start.
#     """
#     transient_states = [s for s in ALL_STATES if s not in [CONVERSION, NULL]]
#     absorbing_states = [CONVERSION, NULL]

#     Q = P.loc[transient_states, transient_states].to_numpy()
#     R = P.loc[transient_states, absorbing_states].to_numpy()

#     I = np.eye(len(transient_states))
#     N = np.linalg.inv(I - Q)   # fundamental matrix
#     B = N @ R                  # absorption probabilities

#     start_idx = transient_states.index(START)
#     conv_idx = absorbing_states.index(CONVERSION)
#     return float(B[start_idx, conv_idx])

def conversion_probability_from_start(P: pd.DataFrame) -> float:
    """
    Compute absorption probability into Conversion starting from Start.
    Works even after a channel has been removed.
    """
    states = list(P.index)
    transient_states = [s for s in states if s not in [CONVERSION, NULL]]
    absorbing_states = [s for s in states if s in [CONVERSION, NULL]]

    Q = P.loc[transient_states, transient_states].to_numpy()
    R = P.loc[transient_states, absorbing_states].to_numpy()

    I = np.eye(len(transient_states))
    N = np.linalg.inv(I - Q)
    B = N @ R

    start_idx = transient_states.index(START)
    conv_idx = absorbing_states.index(CONVERSION)
    return float(B[start_idx, conv_idx])

# Removal effect

In [46]:
def remove_channel_from_path(path: list[str], channel: str) -> list[str]:
    """
    Remove all occurrences of a channel from a path while preserving order.
    """
    new_path = [x for x in path if x != channel]

    # ensure terminal exists
    if len(new_path) == 0 or new_path[-1] not in [CONVERSION, NULL]:
        new_path = new_path + [NULL]

    return new_path


def removal_effects(paths_df: pd.DataFrame, channels: list[str]) -> pd.DataFrame:
    """
    Compute Markov removal effects based on drop in conversion probability.
    """
    base_counts = build_transition_counts(paths_df)
    base_P = build_transition_matrix(base_counts)
    base_conv_prob = conversion_probability_from_start(base_P)

    print("base_counts: \n", base_counts)
    print("base_P: \n", base_P)
    print("base_conv_prob: \n", base_conv_prob)

    rows = []

    for ch in channels:
        mod_df = paths_df.copy()
        mod_df["path"] = mod_df["path"].apply(lambda p: remove_channel_from_path(p, ch))

        mod_counts = build_transition_counts(mod_df)
        mod_P = build_transition_matrix(mod_counts)
        mod_conv_prob = conversion_probability_from_start(mod_P)

        effect = base_conv_prob - mod_conv_prob
        rows.append(
            {
                "channel": ch,
                "base_conv_prob": base_conv_prob,
                "removed_conv_prob": mod_conv_prob,
                "removal_effect": effect,
            }
        )

    out = pd.DataFrame(rows).sort_values("removal_effect", ascending=False).reset_index(drop=True)
    return out

def removal_effect_transition_matrix(P: pd.DataFrame, channel: str) -> pd.DataFrame:
    """
    Remove a channel from the Markov chain by redirecting inbound probability
    mass through the removed channel's outbound distribution.

    For each predecessor i:
        P_new(i, j) = P(i, j) + P(i, channel) * P(channel, j)

    Then remove the channel row/column and renormalize transient rows.
    """
    P_new = P.copy()

    if channel not in P_new.index or channel not in P_new.columns:
        raise ValueError(f"Channel {channel} not found in transition matrix.")

    outbound = P_new.loc[channel, :].copy()

    # Redirect inbound flow through channel to its downstream destinations
    for src in P_new.index:
        if src == channel:
            continue

        p_to_channel = P_new.loc[src, channel]
        if p_to_channel > 0:
            for dst in P_new.columns:
                if dst == channel:
                    continue
                P_new.loc[src, dst] += p_to_channel * outbound[dst] * 2

            P_new.loc[src, channel] = 0.0

    # Drop the channel row and column
    P_new = P_new.drop(index=channel, columns=channel)

    # Renormalize non-absorbing rows
    for src in P_new.index:
        if src in [CONVERSION, NULL]:
            continue
        # By definition p(no convs) increases when dropping a channel.
        P_new.loc[src, NULL] *= 1.2
        row_sum = P_new.loc[src, :].sum()
        if row_sum > 0:
            P_new.loc[src, :] /= row_sum
        else:
            # dead-end goes to Null
            P_new.loc[src, :] = 0.0
            P_new.loc[src, NULL] = 1.0

    # absorbing states
    if CONVERSION in P_new.index:
        P_new.loc[CONVERSION, :] = 0.0
        P_new.loc[CONVERSION, CONVERSION] = 1.0
    if NULL in P_new.index:
        P_new.loc[NULL, :] = 0.0
        P_new.loc[NULL, NULL] = 1.0

    return P_new

def removal_effects_from_matrix(P: pd.DataFrame, channels: list[str]) -> pd.DataFrame:
    base_conv_prob = conversion_probability_from_start(P)

    rows = []
    for ch in channels:
        P_removed = removal_effect_transition_matrix(P, ch)
        removed_conv_prob = conversion_probability_from_start(P_removed)
        effect = base_conv_prob - removed_conv_prob

        rows.append({
            "channel": ch,
            "base_conv_prob": base_conv_prob,
            "removed_conv_prob": removed_conv_prob,
            "removal_effect": effect,
        })

    return pd.DataFrame(rows).sort_values("removal_effect", ascending=False).reset_index(drop=True)

# Normalize into attribution credits

In [47]:
def attribution_from_removal_effects(removal_df: pd.DataFrame, total_conversions: float) -> pd.DataFrame:
    df = removal_df.copy()
    total_effect = df["removal_effect"].clip(lower=0).sum()

    if total_effect <= 0:
        df["attribution_share"] = 0.0
    else:
        df["attribution_share"] = df["removal_effect"].clip(lower=0) / total_effect

    df["attributed_conversions"] = df["attribution_share"] * total_conversions
    return df

# End-to-end example

In [48]:
RNG = np.random.default_rng(42)

if __name__ == "__main__":
    # paths_df = simulate_paths(n_paths=50000, max_steps=5, rng=RNG)

    # pd.set_option('display.max_colwidth', None)
    # print(paths_df.head(20))
    # print("Observed conversion rate:", paths_df["converted"].mean())

    # removal_df = removal_effects(paths_df, CHANNELS)
    # print("\nRemoval effects:")
    # print(removal_df)

    # total_conversions = paths_df["converted"].sum()
    # attrib_df = attribution_from_removal_effects(removal_df, total_conversions)
    # print("\nAttribution results:")
    # print(attrib_df[["channel", "attribution_share", "attributed_conversions"]])

    terminal_probs = {
        "Display": {"Conversion": 0.0171, "Null": 0.6},
        "Search":  {"Conversion": 0.0225,  "Null": 0.75},
        "Social":  {"Conversion": 0.0099,  "Null": 0.45},
        "Email":   {"Conversion": 0.021,  "Null": 0.7},
    }

    trans_probs = {
        "Display": {
            "Display": 0.15,
            "Search": 0.4,
            "Social": 0.25,
            "Email": 0.2,
        },
    
        "Social": {
            "Display": 0.18,
            "Search": 0.50,
            "Social": 0.07,
            "Email": 0.25,
        },
    
        "Search": {
            "Display": 0.02,
            "Search": 0.18,
            "Social": 0.02,
            "Email": 0.78,
        },
    
        "Email": {
            "Display": 0.02,
            "Search": 0.60,
            "Social": 0.03,
            "Email": 0.35,
        },
    }
        
    paths_df = simulate_paths(n_paths=50_000, max_steps=20, rng=RNG, terminal_probs=terminal_probs, trans_probs = trans_probs)

    count_df = build_transition_counts(paths_df)
    P = build_transition_matrix(count_df)
    
    print("base_conv_prob:", conversion_probability_from_start(P))
    
    removal_df = removal_effects_from_matrix(P, CHANNELS)
    print(removal_df)

    total_conversions = paths_df["converted"].sum()
    attrib_df = attribution_from_removal_effects(removal_df, total_conversions)
    print("\nAttribution results:")
    print(attrib_df[["channel", "attribution_share", "attributed_conversions"]])

base_conv_prob: 0.02824
   channel  base_conv_prob  removed_conv_prob  removal_effect
0   Social         0.02824           0.023150        0.005090
1    Email         0.02824           0.023700        0.004540
2  Display         0.02824           0.023723        0.004517
3   Search         0.02824           0.024128        0.004112

Attribution results:
   channel  attribution_share  attributed_conversions
0   Social           0.278753              393.599843
1    Email           0.248628              351.062900
2  Display           0.247405              349.335608
3   Search           0.225214              318.001650
